# MNIST — MLP deneylerini çalıştırma
Bu defter bir çalıştırma rehberidir; hücre çıktıları önceden çalıştırılmış gibi gösterilmemiştir. Gerçek tamamlanmış deneyler `results/main/` içindedir. CNN ve hazır model kullanılmaz.

Yerelde defteri proje kökünden açın. Colab kullanıyorsanız aşağıdaki yükleme hücresine ZIP paketini seçin. Gerçek el yazısı sonuçları `results/handwriting/` klasöründedir; 11/11 rakam doğru tanınmıştır.

In [ ]:
# Yalnızca Colab: ZIP paketini yükleyin. Yerelde bu hücreyi atlayın.
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith(".zip"))
with zipfile.ZipFile(archive) as z:
    z.extractall("/content/project")
os.chdir("/content/project/mnist-mlp")
print(Path.cwd())

In [ ]:
%pip install -r requirements.txt

In [ ]:
from pathlib import Path
assert Path("configs/experiments.json").exists(), "Proje köküne geçin."
import json, torch
print("PyTorch:",torch.__version__,"CUDA:",torch.cuda.is_available())

## 1. Deney planını inceleme
50.000 train, 10.000 validation; resmî 10.000 test yalnızca model seçimi dondurulduktan sonra değerlendirilir.

In [ ]:
config=json.loads(Path("configs/experiments.json").read_text())
config

## 2. Yeniden eğitim
Mevcut sonuçlar korunur. Yeni koşu için boş çıktı klasörü kullanılır. Aşağıdaki komut gerçek eğitim yapar ve zaman alır.

In [ ]:
import subprocess, sys
from datetime import datetime
run_dir="results/notebook_"+datetime.now().strftime("%Y%m%d_%H%M%S")
subprocess.run([sys.executable,"-m","src.train","--output",run_dir],check=True)
subprocess.run([sys.executable,"-m","src.report","--results",run_dir],check=True)

## 3. Mevcut gerçek sonuçları inceleme
Yeniden eğitim yapmadan da bu hücreleri çalıştırabilirsiniz. Yeni koşuyu incelemek için `result_dir=Path(run_dir)` kullanın.

In [ ]:
from IPython.display import display, Markdown, Image
result_dir=Path("results/main")
display(Markdown((result_dir/"RAPOR.md").read_text()))
display(Image(filename=str(result_dir/"comparison.png")))
display(Image(filename=str(result_dir/"confusion_matrix.png")))

## 4. Gerçek el yazısı
Özgün fotoğraf, 11 kırpım ve etiketler hazırdır. `results/handwriting/predictions.json` gerçek sonuçları içerir. Numara 22120606015, tahmin 22120606015, doğruluk 11/11. Yeniden değerlendirme için boş bir çıktı klasörü seçin.

In [ ]:
selection=json.loads((result_dir/"selection.json").read_text())
checkpoint=result_dir/selection["checkpoint"]
print("El yazısı için seçilen model:",checkpoint)
# Paketle gelen gerçek sonucu göster; yeniden çalıştırmak için yorum satırındaki komutu kullan:
# subprocess.run([sys.executable,"-m","src.handwriting","--checkpoint",str(checkpoint),"--output","results/handwriting_notebook"],check=True)
display(Image(filename="results/handwriting/predictions.png"))

## 5. Teslim
El yazısı raporu ve orijinal görseller pakettedir; `docs/TESLIM.md` listesini tamamlayın. Colab oturumu kapanmadan yeni `results` klasörünüzü indirip depoya ekleyin.

In [ ]:
# Colab: yeni sonuçları indirmek için yorumları kaldırın.
# import shutil
# shutil.make_archive("mnist-results", "zip", "results")
# files.download("mnist-results.zip")